# nanoloop-moe quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertlieyingadrian/nanoloop-moe/blob/main/notebooks/quickstart.ipynb)

A small looped mixture-of-experts transformer in the style of Inkling 2 Mini (an Inkling-style toy; not affiliated with Thinking Machines). This notebook trains it briefly on the chain task, changes the number of core passes K at inference, and runs the lens comparison. On a Colab GPU each cell takes about a minute; on a CPU, lower `STEPS`.


In [ ]:
import os, subprocess, sys
# Run from the repo root: clone it on Colab, or step up from notebooks/ when run locally.
if os.path.exists('../nanoloop_moe.py'):
    os.chdir('..')
elif not os.path.exists('nanoloop_moe.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/albertlieyingadrian/nanoloop-moe'], check=True)
    os.chdir('nanoloop-moe')
print(os.getcwd())

In [ ]:
import torch
STEPS = int(os.environ.get('NB_STEPS', 1500 if torch.cuda.is_available() else 40))
SMALL = [] if torch.cuda.is_available() else ['--n_embd', '32', '--n_head', '2', '--hops', '3', '--block_size', '40', '--batch_size', '16']
def run(*args):
    out = subprocess.run([sys.executable, 'nanoloop_moe.py', *args], capture_output=True, text=True)
    print(out.stdout[-3000:], out.stderr[-2000:])
    out.check_returncode()

## 1. The tests (a few seconds on CPU)

In [ ]:
out = subprocess.run([sys.executable, 'test_nanoloop_moe.py'], capture_output=True, text=True)
print(out.stdout, out.stderr[-2000:])
out.check_returncode()

## 2. Train the looped MoE and the plain loop (one residual stream) on the chain task

`compare --only` trains a subset of the five models. The full comparison is `python nanoloop_moe.py --mode compare`.

In [ ]:
run('--mode', 'compare', '--only', 'plainloop,loopmoe', '--steps', str(STEPS), *SMALL)

## 3. Change K at inference

The same weights, run with 1 core pass and with 3.

In [ ]:
run('--mode', 'sample', '--out', 'out/chain_loopmoe.pt', '--loops', '1')
run('--mode', 'sample', '--out', 'out/chain_loopmoe.pt', '--loops', '3')

## 4. Lens: logit lens on the active block vs the mix lens

There is no single residual stream, so each visit offers several states to read. `act_in` is the active block (the plain logit lens), `mlp_in` is the visit's own MoE-input mix, and `mix` is the output head's `final_input` mix applied early.

In [ ]:
run('--mode', 'lens')
run('--mode', 'plot')
from IPython.display import Image, display
if os.path.exists('assets/lens.png'):
    display(Image('assets/lens.png'))